BÀI THI GIỮA KỲ - ĐƯỜNG ỐNG DỮ LIỆU THỜI TIẾT TẠI HÀ NỘI

#**0.Cài đặt môi trường và Khởi tạo (1 Điểm)**
**Mục đích:** Giúp code chạy mượt mà trên cả Colab (của sinh viên) và Ubuntu ảo (của GitHub Actions).

In [ ]:
# ==========================================
# CÀI ĐẶT MÔI TRƯỜNG
# ==========================================
import os
import requests
import json
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("⏳ Đang khởi tạo SparkSession...")
spark = SparkSession.builder.appName("Midterm_BigData_Pipeline").getOrCreate()

# Cấu hình đường dẫn lưu trữ thông minh:
# Nếu chạy trên Colab -> Mount Google Drive. Nếu chạy trên GitHub Actions -> Dùng thư mục ảo /tmp
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/Midterm_BigData"
    print("✅ Đã kết nối Google Drive!")
except ImportError:
    BASE_DIR = "/tmp/Midterm_BigData"
    print("⚠️ Đang chạy trên môi trường Test (GitHub Actions).")

os.makedirs(BASE_DIR, exist_ok=True)
print(f"📂 Thư mục làm việc: {BASE_DIR}")

# ==========================================
# AUTO-TEST 0 (Không được xoá hoặc sửa code đoạn này)
assert spark is not None, "LỖI: Chưa khởi tạo được SparkSession!"
print("🎉 PASSED PHẦN 0 (1 Điểm)")
# ==========================================

⏳ Đang khởi tạo SparkSession...
Mounted at /content/drive
✅ Đã kết nối Google Drive!
📂 Thư mục làm việc: /content/drive/MyDrive/Midterm_BigData
🎉 PASSED PHẦN 0 (1 Điểm)


#**1.Tầng Ingestion - Thu thập dữ liệu (2.5 Điểm)**
**Yêu cầu:** Sinh viên gọi API thời tiết, lấy dữ liệu JSON và lưu xuống file thô (Raw Data)

In [ ]:
# ==========================================
# PHẦN 1: INGESTION - LẤY DỮ LIỆU TỪ API
# ==========================================
def fetch_weather_data(output_filepath):
    """
    YÊU CẦU:
    1. Dùng thư viện requests gọi GET tới URL sau:
       https://api.open-meteo.com/v1/forecast?latitude=21.0285&longitude=105.8542&current=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=Asia%2FBangkok
    2. Chuyển kết quả lấy được thành định dạng JSON.
    3. Lưu toàn bộ dữ liệu JSON đó vào file có đường dẫn là `output_filepath`.
    """
    # --- BẮT ĐẦU CODE CỦA SINH VIÊN ---

        # 1. Gọi GET tới URL bằng thư viện requests
    url = "https://api.open-meteo.com/v1/forecast?latitude=21.0285&longitude=105.8542&current=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=Asia%2FBangkok"
    response = requests.get(url)

    # 2. Chuyển kết quả thành định dạng JSON
    data = response.json()

    # 3. Lưu toàn bộ dữ liệu vào file JSON theo đường dẫn output_filepath
    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)



    # --- KẾT THÚC CODE CỦA SINH VIÊN ---

    return output_filepath

# ==========================================
# AUTO-TEST 1 (Không được xoá hoặc sửa code đoạn này)
raw_file = os.path.join(BASE_DIR, "raw_weather.json")
fetch_weather_data(raw_file)

assert os.path.exists(raw_file), "LỖI: File JSON chưa được tạo!"
with open(raw_file, 'r') as f:
    data = json.load(f)
assert "current" in data, "LỖI: Dữ liệu JSON không đúng cấu trúc API (Thiếu key 'current')."
assert "temperature_2m" in data["current"], "LỖI: Không tìm thấy thông số nhiệt độ!"
print("🎉 PASSED PHẦN 1 (2.5 Điểm)")
# ==========================================

🎉 PASSED PHẦN 1 (2.5 Điểm)


#**2.Tầng Processing - Xử lý dữ liệu với PySpark (3 Điểm)**
**Yêu cầu:** Sinh viên đọc file JSON vừa tải, làm phẳng dữ liệu (Flatten) và thêm cột phân tích nghiệp vụ.

In [ ]:
# ==========================================
# PHẦN 2: PROCESSING - XỬ LÝ BẰNG PYSPARK
# ==========================================
def process_weather_data(spark_session, input_filepath):
    """
    YÊU CẦU:
    1. Dùng SparkSession đọc file JSON tại `input_filepath` (nhớ bật option multiline=true).
    2. Trích xuất các trường sau từ node 'current' và đổi tên cột cho đẹp:
       - 'current.time' -> 'Thoi_Gian'
       - 'current.temperature_2m' -> 'Nhiet_Do'
       - 'current.relative_humidity_2m' -> 'Do_Am'
    3. Tạo thêm cột mới tên là 'Canh_Bao_Nhiet':
       - Nếu Nhiet_Do > 35 -> Gán là "Nắng nóng gay gắt"
       - Ngược lại -> Gán là "Bình thường"
    4. Trả về DataFrame sau khi xử lý.
    """
    # --- BẮT ĐẦU CODE CỦA SINH VIÊN ---

    from pyspark.sql import functions as F

    # 1. Dùng SparkSession đọc file JSON (bật option multiline=true vì file có format indent)
    df = spark_session.read.option("multiline", "true").json(input_filepath)

    # 2. Trích xuất các trường từ node 'current' và đổi tên cột (Flatten)
    df_flatten = df.select(
        F.col("current.time").alias("Thoi_Gian"),
        F.col("current.temperature_2m").alias("Nhiet_Do"),
        F.col("current.relative_humidity_2m").alias("Do_Am")
    )

    # 3. Tạo thêm cột 'Canh_Bao_Nhiet' với logic điều kiện
    df_final = df_flatten.withColumn(
        "Canh_Bao_Nhiet",
        F.when(F.col("Nhiet_Do") > 35, "Nắng nóng gay gắt")
         .otherwise("Bình thường")
    )

    # 4. Trả về DataFrame cuối cùng
    return df_final


    # --- KẾT THÚC CODE CỦA SINH VIÊN ---

# ==========================================
# AUTO-TEST 2 (Không được xoá hoặc sửa code đoạn này)
df_processed = process_weather_data(spark, raw_file)
assert df_processed is not None, "LỖI: Hàm chưa trả về DataFrame!"

cols = df_processed.columns
assert "Thoi_Gian" in cols and "Nhiet_Do" in cols and "Do_Am" in cols, "LỖI: Thiếu cột hoặc đặt sai tên cột!"
assert "Canh_Bao_Nhiet" in cols, "LỖI: Chưa tạo cột Canh_Bao_Nhiet!"
assert df_processed.count() > 0, "LỖI: DataFrame đang trống!"
print("🎉 PASSED PHẦN 2 (3 Điểm)")
# ==========================================

🎉 PASSED PHẦN 2 (3 Điểm)


#**3. Tầng Storage - Lưu trữ dữ liệu sạch (1.5 Điểm)**
**Yêu cầu:** Ghi đè DataFrame đã xử lý thành định dạng CSV để Looker Studio có thể đọc được

In [ ]:
# ==========================================
# PHẦN 3: STORAGE - LƯU TRỮ DỮ LIỆU
# ==========================================
def save_to_datalake(df, output_dir):
    """
    YÊU CẦU:
    Ghi DataFrame `df` ra thư mục `output_dir` dưới dạng file CSV.
    Yêu cầu bật header và chọn chế độ ghi đè (overwrite).
    """
    # --- BẮT ĐẦU CODE CỦA SINH VIÊN ---

    df.write.format("csv") \
        .option("header", "true") \
        .mode("overwrite") \
        .save(output_dir)

    # --- KẾT THÚC CODE CỦA SINH VIÊN ---

# ==========================================
# AUTO-TEST 3 (Không được xoá hoặc sửa code đoạn này)
processed_dir = os.path.join(BASE_DIR, "clean_weather_data")
save_to_datalake(df_processed, processed_dir)

assert os.path.exists(processed_dir), "LỖI: Thư mục output chưa được tạo!"
csv_files = [f for f in os.listdir(processed_dir) if f.endswith('.csv')]
assert len(csv_files) > 0, "LỖI: Không tìm thấy file CSV nào được tạo ra!"
print("🎉 PASSED PHẦN 3 (1.5 Điểm)")
# ==========================================

🎉 PASSED PHẦN 3 (1.5 Điểm)


#**4. Tầng Visualization - Báo cáo Looker Studio (2 Điểm)**
**Lưu ý:** Phần này Autograding chỉ cho điểm tạm thời để kiểm tra xem sinh viên có nộp link hay không. Giảng viên sẽ click vào link để chấm độ thẩm mỹ và tính chính xác của biểu đồ.

In [ ]:
# ==========================================
# PHẦN 4: VISUALIZATION - LOOKER STUDIO
# ==========================================
"""
HƯỚNG DẪN DÀNH CHO SINH VIÊN:
1. Vào thư mục 'Midterm_BigData/clean_weather_data' trên Google Drive của bạn.
2. Tải file .csv bên trong thư mục đó về máy.
3. Truy cập https://lookerstudio.google.com/ -> Tạo Báo cáo trống.
4. Chọn nguồn dữ liệu là "Tải tệp lên" (File Upload) và upload file csv vừa tải.
5. Vẽ ít nhất 2 biểu đồ:
   - 1 Thẻ điểm (Scorecard) hiển thị Nhiệt độ trung bình.
   - 1 Biểu đồ chuỗi thời gian (Time series) hoặc Tròn (Pie chart) đánh giá Độ ẩm/Cảnh báo.
6. Bấm "Chia sẻ", bật quyền "Bất kỳ ai có liên kết đều có thể xem".
7. Dán đường link báo cáo của bạn vào biến dưới đây:
"""

# --- SINH VIÊN DÁN LINK VÀO ĐÂY ---
LOOKER_STUDIO_LINK = "https://lookerstudio.google.com/reporting/732e6246-fd13-459b-9a08-54b428410750"
# ----------------------------------


# ==========================================
# AUTO-TEST 4 (Không được xoá hoặc sửa code đoạn này)
assert LOOKER_STUDIO_LINK != "https://lookerstudio.google.com/reporting/...", "LỖI: Bạn chưa dán link Looker Studio!"
assert LOOKER_STUDIO_LINK.startswith("https://lookerstudio.google.com/reporting/"), "LỖI: Link không hợp lệ!"
print("🎉 PASSED PHẦN 4 (2 Điểm). Đã nộp báo cáo thành công!")
print("\n✅ CHÚC MỪNG BẠN ĐÃ HOÀN THÀNH BÀI THI GIỮA KỲ!")

🎉 PASSED PHẦN 4 (2 Điểm). Đã nộp báo cáo thành công!

✅ CHÚC MỪNG BẠN ĐÃ HOÀN THÀNH BÀI THI GIỮA KỲ!
